# CellGraph — a learned model of the whole cell

A graph neural network (SIGN/SGC) over the 16,492-node multi-relational cell knowledge graph.
Answers: what binds what · remove protein X → downstream · drug → off-targets · does wiring encode
function. CPU-only, a few minutes. See `docs/CELLGRAPH.md`.


## 1 · Clone + install


In [ ]:
import os, sys
BR='claude/vectorize-gex-propensity-zp09w8'
if not os.path.exists('colab/cellgraph.py'):
    os.system(f'git clone -q --branch {BR} https://github.com/nikku03/cell.git')
    if os.path.isdir('cell'): os.chdir('cell')
os.system('pip -q install numpy scipy scikit-learn')
sys.path.insert(0,'colab'); os.makedirs('outputs/orphan', exist_ok=True)


## 2 · Restore the deeper model from Drive


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import glob, gzip, shutil, json
dst='outputs/orphan/cell_complete.json'
if not os.path.exists(dst):
    c=sorted(glob.glob('/content/drive/MyDrive/cell_model/**/cell_complete*.json*',recursive=True),
             key=lambda p: os.path.getsize(p), reverse=True)
    src=c[0]; print('using', src)
    (shutil.copyfileobj(gzip.open(src,'rb'),open(dst,'wb')) if src.endswith('.gz') else shutil.copy(src,dst))
D=json.load(open(dst)); print('cell types', len(D['ctnames']), '| emask', len(D['emask']))


## 3 · Train embeddings + the four capability metrics


In [ ]:
import subprocess
print(subprocess.run([sys.executable,'colab/cellgraph.py'],capture_output=True,text=True).stdout)
print(subprocess.run([sys.executable,'colab/validate_cellgraph.py'],capture_output=True,text=True).stdout)


## 4 · Live queries — the model answering mechanistic questions


In [ ]:
from cellgraph import CellGraph
cg = CellGraph()
print('TP53 can bind      ->', cg.bind_partners('TP53', 10))
print('remove SREBF2      ->', cg.knockout_effect('SREBF2', 10))
print('remove TP53        ->', cg.knockout_effect('TP53', 10))
print('Imatinib off-target->', cg.drug_off_targets('Imatinib', 10))
print('what binds EGFR    ->', cg.bind_partners('EGFR', 10))


## Scale path (next rounds)
1. **Learned GNN (torch GraphSAGE/R-GCN)** to beat the fixed-propagation link AUC.
2. **Supervised perturbation on the real Replogle screen** — calibrate magnitude, not just direction.
3. **Fold in Geneformer / scGPT / Tahoe embeddings** (on Drive) as extra node features.
